# 01 — Ingestion and Raw Validation

## Tujuan

Notebook ini bertanggung jawab untuk:

1. Memvalidasi source file.
2. Mengambil fingerprint source menggunakan SHA-256.
3. Membaca raw dataset.
4. Menstandarkan nama kolom ke lowercase.
5. Memvalidasi schema dan data type.
6. Memvalidasi primary key.
7. Memvalidasi duplicate record.
8. Membuat profiling missing values.
9. Memvalidasi kualitas string.
10. Memvalidasi domain numerik.
11. Memvalidasi domain kategorikal.
12. Memvalidasi hubungan train/test.
13. Menyimpan dataset ke staged Parquet.
14. Memvalidasi ulang staged artifact.
15. Membuat validation report.
16. Membuat ingestion metadata.
17. Mencatat provenance dan environment.
18. Membersihkan resource setelah proses selesai.

## Boundary

Notebook ini **tidak melakukan data cleaning substantif**.

Tidak dilakukan:

- imputasi missing value
- penghapusan duplicate
- penghapusan outlier
- encoding
- feature engineering
- perubahan nilai data
- koreksi data source

Normalisasi nama kolom ke lowercase hanya merupakan standardisasi schema teknis.

Jika ditemukan masalah pada raw dataset, masalah tersebut **dideteksi dan dilaporkan**, bukan diperbaiki di notebook ini.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import gc
import platform
import subprocess
import sys
import time

import polars as pl

In [2]:
# Set global session limits to display everything
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

# ============================================================
# PATH CONFIGURATION
# ============================================================

TRAIN_RAW = Path("../data/raw/train.csv")
TEST_RAW = Path("../data/raw/test.csv")

STAGED_DIR = Path("../data/staged")
METADATA_DIR = STAGED_DIR / "metadata"

TRAIN_PARQUET = STAGED_DIR / "train.parquet"
TEST_PARQUET = STAGED_DIR / "test.parquet"

INGESTION_METADATA = METADATA_DIR / "ingestion_metadata.json"
VALIDATION_REPORT = METADATA_DIR / "validation_report.json"


STAGED_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)


print("TRAIN_RAW :", TRAIN_RAW)
print("TEST_RAW  :", TEST_RAW)
print("STAGED_DIR :", STAGED_DIR)
print("METADATA  :", METADATA_DIR)

TRAIN_RAW : ..\data\raw\train.csv
TEST_RAW  : ..\data\raw\test.csv
STAGED_DIR : ..\data\staged
METADATA  : ..\data\staged\metadata


In [3]:
# ============================================================
# PIPELINE CONTRACT
# ============================================================

DATASET_NAME = "titanic"

PIPELINE_STAGE = "ingestion"

PIPELINE_VERSION = "1.0.0"

SCHEMA_VERSION = "1.0.0"

DQ_RULES_VERSION = "1.0.0"

SOURCE_FORMAT = "csv"

ARTIFACT_FORMAT = "parquet"


print("Dataset          : ", DATASET_NAME)
print("Pipeline stage   : ", PIPELINE_STAGE)
print("Pipeline version : ", PIPELINE_VERSION)
print("Schema version   : ", SCHEMA_VERSION)
print("DQ rules version : ", DQ_RULES_VERSION)

Dataset          :  titanic
Pipeline stage   :  ingestion
Pipeline version :  1.0.0
Schema version   :  1.0.0
DQ rules version :  1.0.0


In [4]:
# ============================================================
# RUN IDENTIFICATION
# ============================================================

INGESTION_START = time.perf_counter()

RUN_TIMESTAMP = datetime.now(timezone.utc)

RUN_TIMESTAMP_COMPACT = RUN_TIMESTAMP.strftime("%Y%m%dT%H%M%SZ")

RUN_ID = f"{DATASET_NAME}-{PIPELINE_STAGE}-{RUN_TIMESTAMP_COMPACT}"


print("Run ID:", RUN_ID)
print("Started:", RUN_TIMESTAMP.isoformat())

Run ID: titanic-ingestion-20260917T034007Z
Started: 2026-09-17T03:40:07.822799+00:00


In [5]:
# ============================================================
# EXPECTED SCHEMA
# ============================================================

EXPECTED_TRAIN_COLUMNS = [
    "passengerid",
    "survived",
    "pclass",
    "name",
    "sex",
    "age",
    "sibsp",
    "parch",
    "ticket",
    "fare",
    "cabin",
    "embarked",
]


EXPECTED_TEST_COLUMNS = [
    "passengerid",
    "pclass",
    "name",
    "sex",
    "age",
    "sibsp",
    "parch",
    "ticket",
    "fare",
    "cabin",
    "embarked",
]


EXPECTED_DTYPES = {
    "passengerid": pl.Int64,
    "survived": pl.Int64,
    "pclass": pl.Int64,
    "name": pl.String,
    "sex": pl.String,
    "age": pl.Float64,
    "sibsp": pl.Int64,
    "parch": pl.Int64,
    "ticket": pl.String,
    "fare": pl.Float64,
    "cabin": pl.String,
    "embarked": pl.String,
}

In [6]:
# ============================================================
# DETAILED SCHEMA CONTRACT
# ============================================================

SCHEMA_CONTRACT = {
    "passengerid": {
        "dtype": "Int64",
        "nullable": False,
        "required": True,
        "semantic_type": "primary_key",
    },
    "survived": {
        "dtype": "Int64",
        "nullable": False,
        "required": True,
        "semantic_type": "binary_target",
        "allowed_values": [0, 1],
    },
    "pclass": {
        "dtype": "Int64",
        "nullable": False,
        "required": True,
        "semantic_type": "categorical",
        "allowed_values": [1, 2, 3],
    },
    "name": {
        "dtype": "String",
        "nullable": False,
        "required": True,
        "semantic_type": "text",
    },
    "sex": {
        "dtype": "String",
        "nullable": False,
        "required": True,
        "semantic_type": "categorical",
        "allowed_values": ["male", "female"],
    },
    "age": {
        "dtype": "Float64",
        "nullable": True,
        "required": False,
        "semantic_type": "numeric",
        "min": 0,
        "max": 100,
    },
    "sibsp": {
        "dtype": "Int64",
        "nullable": False,
        "required": True,
        "semantic_type": "count",
        "min": 0,
    },
    "parch": {
        "dtype": "Int64",
        "nullable": False,
        "required": True,
        "semantic_type": "count",
        "min": 0,
    },
    "ticket": {
        "dtype": "String",
        "nullable": False,
        "required": True,
        "semantic_type": "identifier",
    },
    "fare": {
        "dtype": "Float64",
        "nullable": True,
        "required": False,
        "semantic_type": "numeric",
        "min": 0,
    },
    "cabin": {
        "dtype": "String",
        "nullable": True,
        "required": False,
        "semantic_type": "categorical_text",
    },
    "embarked": {
        "dtype": "String",
        "nullable": True,
        "required": False,
        "semantic_type": "categorical",
        "allowed_values": ["C", "Q", "S"],
    },
}


STRING_COLUMNS = [
    "name",
    "sex",
    "ticket",
    "cabin",
    "embarked",
]

In [7]:
# ============================================================
# MISSING VALUE CONTRACT
# ============================================================

# Hanya token yang memang dianggap missing oleh source contract.
#
# Jangan memasukkan token generik seperti "No" secara global
# karena dapat merupakan nilai valid pada dataset lain.

MISSING_VALUES = [
    "",
    "NA",
    "N/A",
    "na",
    "n/a",
    "N/a",
]


MISSINGNESS_THRESHOLDS = {
    "passengerid": 0.00,
    "survived": 0.00,
    "pclass": 0.00,
    "name": 0.00,
    "sex": 0.00,
    "age": 0.30,
    "sibsp": 0.00,
    "parch": 0.00,
    "ticket": 0.00,
    "fare": 0.01,
    "cabin": 0.90,
    "embarked": 0.01,
}

Threshold di atas adalah policy untuk dataset Titanic project ini, bukan aturan universal untuk semua dataset.

In [8]:
# ============================================================
# SHA-256
# ============================================================

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    sha256 = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

In [9]:
# ============================================================
# SOURCE FILE VALIDATION
# ============================================================

def validate_source_file(path: Path) -> dict:
    assert path.exists(), f"File tidak ditemukan: {path}"

    assert path.is_file(), (
        f"Path bukan file: {path}"
    )

    assert path.suffix.lower() == ".csv", (
        f"Source file harus CSV: {path}"
    )

    size_bytes = path.stat().st_size

    assert size_bytes > 0, (
        f"Source file kosong: {path}"
    )

    return {
        "path": str(path),
        "file_name": path.name,
        "extension": path.suffix.lower(),
        "size_bytes": size_bytes,
    }


train_file_info = validate_source_file(TRAIN_RAW)
test_file_info = validate_source_file(TEST_RAW)

print("✓ Source files exist and are valid CSV files")

✓ Source files exist and are valid CSV files


In [10]:
# ============================================================
# SOURCE FINGERPRINT
# ============================================================

train_sha256 = sha256_file(TRAIN_RAW)
test_sha256 = sha256_file(TEST_RAW)


print("Train SHA256:")
print(train_sha256)

print()

print("Test SHA256:")
print(test_sha256)

Train SHA256:
7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f

Test SHA256:
56023b9948236f3c7a1c9448fcf418b283e109ef177fa8c7e069158dd7dd52b2


In [11]:
# ============================================================
# CSV STRUCTURAL VALIDATION
# ============================================================

def validate_csv_structure(path: Path) -> dict:
    raw_bytes = path.read_bytes()

    assert len(raw_bytes) > 0, (
        f"CSV kosong: {path}"
    )

    # UTF-8 validation
    try:
        text = raw_bytes.decode("utf-8-sig")
    except UnicodeDecodeError as exc:
        raise AssertionError(
            f"{path} bukan UTF-8 compatible CSV"
        ) from exc

    lines = text.splitlines()

    assert len(lines) >= 2, (
        f"{path} harus memiliki header dan minimal satu data row"
    )

    header = lines[0]

    assert "," in header, (
        f"{path} tidak terlihat menggunakan delimiter ','"
    )

    header_columns = [
        column.strip().lower()
        for column in header.split(",")
    ]

    duplicate_columns = sorted(
        {
            column
            for column in header_columns
            if header_columns.count(column) > 1
        }
    )

    assert not duplicate_columns, (
        f"{path} memiliki duplicate header: {duplicate_columns}"
    )

    return {
        "encoding": "utf-8",
        "delimiter": ",",
        "line_count": len(lines),
        "header": header_columns,
        "duplicate_headers": duplicate_columns,
    }


train_csv_structure = validate_csv_structure(TRAIN_RAW)
test_csv_structure = validate_csv_structure(TEST_RAW)

print("✓ CSV structural validation passed")

✓ CSV structural validation passed


In [12]:
# ============================================================
# RAW DATA LOADING
# ============================================================

def load_raw_csv(path: Path) -> pl.DataFrame:
    try:
        return pl.read_csv(
            path,
            null_values=MISSING_VALUES,
            try_parse_dates=False,
            infer_schema_length=1000,
        )

    except Exception as exc:
        raise RuntimeError(
            f"Gagal membaca CSV: {path}. "
            f"Periksa encoding, delimiter, quoting, atau struktur CSV. "
            f"Detail: {exc}"
        ) from exc


train = load_raw_csv(TRAIN_RAW)
test = load_raw_csv(TEST_RAW)


print("✓ Raw CSV loaded")
print("Train shape:", train.shape)
print("Test shape :", test.shape)

✓ Raw CSV loaded
Train shape: (891, 12)
Test shape : (418, 11)


In [13]:
# ============================================================
# TECHNICAL COLUMN NAME STANDARDIZATION
# ============================================================

def normalize_column_names(df: pl.DataFrame) -> pl.DataFrame:
    normalized = [
        column.strip().lower()
        for column in df.columns
    ]

    if len(normalized) != len(set(normalized)):
        duplicates = sorted(
            {
                column
                for column in normalized
                if normalized.count(column) > 1
            }
        )

        raise ValueError(
            f"Duplicate column names setelah normalisasi: {duplicates}"
        )

    return df.rename(
        dict(zip(df.columns, normalized))
    )


train = normalize_column_names(train)
test = normalize_column_names(test)


print("✓ Column names normalized")
print("Train columns:")
print(train.columns)

print()

print("Test columns:")
print(test.columns)

✓ Column names normalized
Train columns:
['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']

Test columns:
['passengerid', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']


In [14]:
# ============================================================
# COLUMN VALIDATION
# ============================================================

def validate_columns(
    df: pl.DataFrame,
    expected_columns: list[str],
    dataset_name: str,
):
    assert len(df.columns) == len(set(df.columns)), (
        f"{dataset_name}: duplicate column names"
    )

    assert df.width == len(expected_columns), (
        f"{dataset_name}: jumlah kolom tidak sesuai. "
        f"expected={len(expected_columns)}, "
        f"actual={df.width}"
    )

    assert df.columns == expected_columns, (
        f"{dataset_name}: column schema tidak sesuai.\n"
        f"Expected: {expected_columns}\n"
        f"Actual  : {df.columns}"
    )


validate_columns(
    train,
    EXPECTED_TRAIN_COLUMNS,
    "train",
)

validate_columns(
    test,
    EXPECTED_TEST_COLUMNS,
    "test",
)


print("✓ Column schema valid")

✓ Column schema valid


In [15]:
# ============================================================
# SCHEMA FINGERPRINT
# ============================================================

def schema_fingerprint(
    df: pl.DataFrame,
) -> str:

    schema_definition = [
        {
            "name": column,
            "dtype": str(dtype),
        }
        for column, dtype in df.schema.items()
    ]

    canonical_schema = json.dumps(
        schema_definition,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        canonical_schema.encode("utf-8")
    ).hexdigest()


train_schema_sha256 = schema_fingerprint(train)
test_schema_sha256 = schema_fingerprint(test)


print("Train schema SHA256:")
print(train_schema_sha256)

print()

print("Test schema SHA256:")
print(test_schema_sha256)

Train schema SHA256:
0f3eb72d2000aa4f1d222676c7802f36bb4a945c72879844bbf02df1ee8530d2

Test schema SHA256:
1e26a7abd624b4491dd13e8a5993e76b0fbab542a1355e664077f7f48d7eef16


In [16]:
# ============================================================
# DATA TYPE VALIDATION
# ============================================================

def validate_dtypes(
    df: pl.DataFrame,
    dataset_name: str,
):
    for column, expected_dtype in EXPECTED_DTYPES.items():

        if column not in df.columns:
            continue

        actual_dtype = df.schema[column]

        assert actual_dtype == expected_dtype, (
            f"{dataset_name}.{column}: "
            f"expected={expected_dtype}, "
            f"actual={actual_dtype}"
        )


validate_dtypes(train, "train")
validate_dtypes(test, "test")


print("✓ Data types valid")

✓ Data types valid


In [17]:
# ============================================================
# ROW COUNT VALIDATION
# ============================================================

assert train.height > 0, (
    "Train dataset kosong"
)

assert test.height > 0, (
    "Test dataset kosong"
)


print("✓ Train rows:", train.height)
print("✓ Test rows :", test.height)

✓ Train rows: 891
✓ Test rows : 418


In [18]:
# ============================================================
# PRIMARY KEY VALIDATION
# ============================================================

def validate_identifier(
    df: pl.DataFrame,
    dataset_name: str,
):

    column = "passengerid"

    null_count = df[column].null_count()

    assert null_count == 0, (
        f"{dataset_name}.{column} memiliki "
        f"{null_count} null"
    )

    unique_count = df[column].n_unique()

    assert unique_count == df.height, (
        f"{dataset_name}.{column} memiliki "
        f"duplicate identifier"
    )

    min_value = df[column].min()

    assert min_value is not None and min_value > 0, (
        f"{dataset_name}.{column} harus positif"
    )


validate_identifier(train, "train")
validate_identifier(test, "test")


print("✓ Primary key validation passed")

✓ Primary key validation passed


In [19]:
# ============================================================
# FULL ROW DUPLICATE VALIDATION
# ============================================================

def count_duplicate_rows(
    df: pl.DataFrame,
) -> int:

    return df.height - df.unique().height


train_duplicate_rows = count_duplicate_rows(train)
test_duplicate_rows = count_duplicate_rows(test)


print("Train duplicate rows:", train_duplicate_rows)
print("Test duplicate rows :", test_duplicate_rows)


assert train_duplicate_rows == 0, (
    f"Train memiliki {train_duplicate_rows} duplicate row"
)

assert test_duplicate_rows == 0, (
    f"Test memiliki {test_duplicate_rows} duplicate row"
)


print("✓ Duplicate validation passed")

Train duplicate rows: 0
Test duplicate rows : 0
✓ Duplicate validation passed


In [20]:
# ============================================================
# MISSING VALUE PROFILING
# ============================================================

def missing_report(
    df: pl.DataFrame,
) -> pl.DataFrame:

    if df.height == 0:
        return pl.DataFrame({
            "column": df.columns,
            "null_count": [0] * df.width,
            "null_ratio": [0.0] * df.width,
            "null_percent": [0.0] * df.width,
        })

    return (
        df.null_count()
        .transpose(
            include_header=True,
            header_name="column",
            column_names=["null_count"],
        )
        .with_columns(
            (
                pl.col("null_count") / df.height
            ).alias("null_ratio"),

            (
                pl.col("null_count")
                / df.height
                * 100
            ).round(2).alias("null_percent"),
        )
    )


train_missing = missing_report(train)
test_missing = missing_report(test)


display(train_missing)
display(test_missing)

column,null_count,null_ratio,null_percent
str,u32,f64,f64
"""passengerid""",0,0.0,0.0
"""survived""",0,0.0,0.0
"""pclass""",0,0.0,0.0
"""name""",0,0.0,0.0
"""sex""",0,0.0,0.0
"""age""",177,0.198653,19.87
"""sibsp""",0,0.0,0.0
"""parch""",0,0.0,0.0
"""ticket""",0,0.0,0.0


column,null_count,null_ratio,null_percent
str,u32,f64,f64
"""passengerid""",0,0.0,0.0
"""pclass""",0,0.0,0.0
"""name""",0,0.0,0.0
"""sex""",0,0.0,0.0
"""age""",86,0.205742,20.57
"""sibsp""",0,0.0,0.0
"""parch""",0,0.0,0.0
"""ticket""",0,0.0,0.0
"""fare""",1,0.002392,0.24


In [21]:
# ============================================================
# REQUIRED NON-NULL VALIDATION
# ============================================================

REQUIRED_NON_NULL = {
    "train": [
        "passengerid",
        "survived",
        "pclass",
        "name",
        "sex",
        "sibsp",
        "parch",
        "ticket",
        "fare",
    ],
    "test": [
        "passengerid",
        "pclass",
        "name",
        "sex",
        "sibsp",
        "parch",
        "ticket",
    ],
}


def validate_required_non_null(
    df: pl.DataFrame,
    dataset_name: str,
):

    for column in REQUIRED_NON_NULL[dataset_name]:

        null_count = df[column].null_count()

        assert null_count == 0, (
            f"{dataset_name}.{column} memiliki "
            f"{null_count} null"
        )


validate_required_non_null(train, "train")
validate_required_non_null(test, "test")


print("✓ Required non-null validation passed")

✓ Required non-null validation passed


In [22]:
# ============================================================
# MISSINGNESS THRESHOLD VALIDATION
# ============================================================

def validate_missingness_threshold(
    df: pl.DataFrame,
    dataset_name: str,
):
    results = []

    for column, threshold in MISSINGNESS_THRESHOLDS.items():

        if column not in df.columns:
            continue

        null_count = df[column].null_count()

        ratio = (
            null_count / df.height
            if df.height > 0
            else 0
        )

        status = (
            "PASS"
            if ratio <= threshold
            else "FAIL"
        )

        results.append({
            "dataset": dataset_name,
            "column": column,
            "null_count": null_count,
            "null_ratio": ratio,
            "threshold": threshold,
            "status": status,
        })

    return pl.DataFrame(results)


train_missing_threshold = validate_missingness_threshold(
    train,
    "train",
)

test_missing_threshold = validate_missingness_threshold(
    test,
    "test",
)


display(train_missing_threshold)
display(test_missing_threshold)


assert (
    train_missing_threshold["status"] == "PASS"
).all()

assert (
    test_missing_threshold["status"] == "PASS"
).all()


print("✓ Missingness thresholds valid")

dataset,column,null_count,null_ratio,threshold,status
str,str,i64,f64,f64,str
"""train""","""passengerid""",0,0.0,0.0,"""PASS"""
"""train""","""survived""",0,0.0,0.0,"""PASS"""
"""train""","""pclass""",0,0.0,0.0,"""PASS"""
"""train""","""name""",0,0.0,0.0,"""PASS"""
"""train""","""sex""",0,0.0,0.0,"""PASS"""
"""train""","""age""",177,0.198653,0.3,"""PASS"""
"""train""","""sibsp""",0,0.0,0.0,"""PASS"""
"""train""","""parch""",0,0.0,0.0,"""PASS"""
"""train""","""ticket""",0,0.0,0.0,"""PASS"""


dataset,column,null_count,null_ratio,threshold,status
str,str,i64,f64,f64,str
"""test""","""passengerid""",0,0.0,0.0,"""PASS"""
"""test""","""pclass""",0,0.0,0.0,"""PASS"""
"""test""","""name""",0,0.0,0.0,"""PASS"""
"""test""","""sex""",0,0.0,0.0,"""PASS"""
"""test""","""age""",86,0.205742,0.3,"""PASS"""
"""test""","""sibsp""",0,0.0,0.0,"""PASS"""
"""test""","""parch""",0,0.0,0.0,"""PASS"""
"""test""","""ticket""",0,0.0,0.0,"""PASS"""
"""test""","""fare""",1,0.002392,0.01,"""PASS"""


✓ Missingness thresholds valid


In [23]:
# ============================================================
# STRING QUALITY VALIDATION
# ============================================================

def validate_string_quality(
    df: pl.DataFrame,
    dataset_name: str,
) -> pl.DataFrame:

    results = []

    for column in STRING_COLUMNS:

        if column not in df.columns:
            continue

        non_null = pl.col(column).is_not_null()

        empty_count = df.filter(
            non_null &
            (pl.col(column).str.len_chars() == 0)
        ).height

        whitespace_count = df.filter(
            non_null &
            (
                pl.col(column)
                != pl.col(column).str.strip_chars()
            )
        ).height

        if empty_count > 0:
            status = "FAIL"

        elif whitespace_count > 0:
            status = "WARNING"

        else:
            status = "PASS"

        results.append({
            "dataset": dataset_name,
            "column": column,
            "empty_string_count": empty_count,
            "whitespace_count": whitespace_count,
            "status": status,
        })

    return pl.DataFrame(results)


train_string_quality = validate_string_quality(
    train,
    "train",
)

test_string_quality = validate_string_quality(
    test,
    "test",
)


display(train_string_quality)
display(test_string_quality)

dataset,column,empty_string_count,whitespace_count,status
str,str,i64,i64,str
"""train""","""name""",0,2,"""WARNING"""
"""train""","""sex""",0,0,"""PASS"""
"""train""","""ticket""",0,0,"""PASS"""
"""train""","""cabin""",0,0,"""PASS"""
"""train""","""embarked""",0,0,"""PASS"""


dataset,column,empty_string_count,whitespace_count,status
str,str,i64,i64,str
"""test""","""name""",0,2,"""WARNING"""
"""test""","""sex""",0,0,"""PASS"""
"""test""","""ticket""",0,0,"""PASS"""
"""test""","""cabin""",0,0,"""PASS"""
"""test""","""embarked""",0,0,"""PASS"""


In [24]:
# ============================================================
# STRING QUALITY DETAILS
# ============================================================

def show_whitespace_issues(
    df: pl.DataFrame,
    dataset_name: str,
    column: str,
    limit: int = 20,
):

    if column not in df.columns:
        return

    issues = df.filter(
        pl.col(column).is_not_null()
        &
        (
            pl.col(column)
            != pl.col(column).str.strip_chars()
        )
    )

    if issues.height == 0:
        return

    print(
        f"⚠ {dataset_name}.{column}: "
        f"{issues.height} row memiliki leading/trailing whitespace"
    )

    display(
        issues
        .select([
            "passengerid",
            column,
        ])
        .head(limit)
    )


for column in STRING_COLUMNS:
    show_whitespace_issues(
        train,
        "train",
        column,
    )


for column in STRING_COLUMNS:
    show_whitespace_issues(
        test,
        "test",
        column,
    )

⚠ train.name: 2 row memiliki leading/trailing whitespace


passengerid,name
i64,str
16,"""Hewlett, Mrs. (Mary D Kingcome…"
858,"""Daly, Mr. Peter Denis """


⚠ test.name: 2 row memiliki leading/trailing whitespace


passengerid,name
i64,str
1167,"""Bryhl, Miss. Dagmar Jenny Inge…"
1222,"""Davies, Mrs. John Morgan (Eliz…"


In [25]:
# ============================================================
# NUMERIC DOMAIN VALIDATION
# ============================================================

def validate_numeric_domains(
    df: pl.DataFrame,
    dataset_name: str,
):

    if "age" in df.columns:

        invalid_age = df.filter(
            pl.col("age").is_not_null()
            &
            (
                (pl.col("age") < 0)
                |
                (pl.col("age") > 100)
            )
        )

        assert invalid_age.height == 0, (
            f"{dataset_name}.age memiliki "
            f"{invalid_age.height} nilai di luar 0..100"
        )


    if "fare" in df.columns:

        invalid_fare = df.filter(
            pl.col("fare").is_not_null()
            &
            (pl.col("fare") < 0)
        )

        assert invalid_fare.height == 0, (
            f"{dataset_name}.fare memiliki "
            f"{invalid_fare.height} nilai negatif"
        )


    for column in ["sibsp", "parch"]:

        if column not in df.columns:
            continue

        invalid_count = df.filter(
            pl.col(column).is_not_null()
            &
            (pl.col(column) < 0)
        )

        assert invalid_count.height == 0, (
            f"{dataset_name}.{column} memiliki "
            f"{invalid_count.height} nilai negatif"
        )


validate_numeric_domains(train, "train")
validate_numeric_domains(test, "test")


print("✓ Numeric domain validation passed")

✓ Numeric domain validation passed


In [26]:
# ============================================================
# CATEGORICAL DOMAIN VALIDATION
# ============================================================

ALLOWED_VALUES = {
    "survived": {0, 1},
    "pclass": {1, 2, 3},
    "sex": {"male", "female"},
    "embarked": {"C", "Q", "S"},
}


def validate_categorical_domains(
    df: pl.DataFrame,
    dataset_name: str,
):

    for column, allowed in ALLOWED_VALUES.items():

        if column not in df.columns:
            continue

        observed = set(
            df[column]
            .drop_nulls()
            .unique()
            .to_list()
        )

        invalid = observed - allowed

        assert not invalid, (
            f"{dataset_name}.{column} memiliki "
            f"nilai invalid: {sorted(invalid)}"
        )


validate_categorical_domains(
    train,
    "train",
)

validate_categorical_domains(
    test,
    "test",
)


print("✓ Categorical domain validation passed")

✓ Categorical domain validation passed


In [27]:
# ============================================================
# TRAIN / TEST RELATIONSHIP
# ============================================================

train_ids = set(
    train["passengerid"].to_list()
)

test_ids = set(
    test["passengerid"].to_list()
)


overlap_ids = train_ids & test_ids


print("Train IDs:", len(train_ids))
print("Test IDs :", len(test_ids))
print("Overlap  :", len(overlap_ids))


assert not overlap_ids, (
    f"Train/Test passengerid overlap: "
    f"{len(overlap_ids)} IDs"
)


print("✓ Train/Test relationship valid")

Train IDs: 891
Test IDs : 418
Overlap  : 0
✓ Train/Test relationship valid


In [28]:
# ============================================================
# RAW DATASET SUMMARY
# ============================================================

def dataset_summary(
    df: pl.DataFrame,
    dataset_name: str,
) -> dict:

    return {
        "dataset": dataset_name,
        "rows": df.height,
        "columns": df.width,
        "column_names": df.columns,
        "schema": {
            column: str(dtype)
            for column, dtype in df.schema.items()
        },
        "schema_sha256": schema_fingerprint(df),
        "estimated_size_bytes": df.estimated_size(),
    }


train_summary = dataset_summary(
    train,
    "train",
)

test_summary = dataset_summary(
    test,
    "test",
)


print(json.dumps(
    train_summary,
    indent=2,
    ensure_ascii=False,
))

print()

print(json.dumps(
    test_summary,
    indent=2,
    ensure_ascii=False,
))

{
  "dataset": "train",
  "rows": 891,
  "columns": 12,
  "column_names": [
    "passengerid",
    "survived",
    "pclass",
    "name",
    "sex",
    "age",
    "sibsp",
    "parch",
    "ticket",
    "fare",
    "cabin",
    "embarked"
  ],
  "schema": {
    "passengerid": "Int64",
    "survived": "Int64",
    "pclass": "Int64",
    "name": "String",
    "sex": "String",
    "age": "Float64",
    "sibsp": "Int64",
    "parch": "Int64",
    "ticket": "String",
    "fare": "Float64",
    "cabin": "String",
    "embarked": "String"
  },
  "schema_sha256": "0f3eb72d2000aa4f1d222676c7802f36bb4a945c72879844bbf02df1ee8530d2",
  "estimated_size_bytes": 85871
}

{
  "dataset": "test",
  "rows": 418,
  "columns": 11,
  "column_names": [
    "passengerid",
    "pclass",
    "name",
    "sex",
    "age",
    "sibsp",
    "parch",
    "ticket",
    "fare",
    "cabin",
    "embarked"
  ],
  "schema": {
    "passengerid": "Int64",
    "pclass": "Int64",
    "name": "String",
    "sex": "String",


In [29]:
# ============================================================
# RAW DATA PREVIEW
# ============================================================

print("TRAIN")
display(train.head(5))

print("TEST")
display(test.head(5))

TRAIN


passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


TEST


passengerid,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
i64,i64,str,str,f64,i64,i64,str,f64,str,str
892,3,"""Kelly, Mr. James""","""male""",34.5,0,0,"""330911""",7.8292,null,"""Q"""
893,3,"""Wilkes, Mrs. James (Ellen Need…","""female""",47.0,1,0,"""363272""",7.0,null,"""S"""
894,2,"""Myles, Mr. Thomas Francis""","""male""",62.0,0,0,"""240276""",9.6875,null,"""Q"""
895,3,"""Wirz, Mr. Albert""","""male""",27.0,0,0,"""315154""",8.6625,null,"""S"""
896,3,"""Hirvonen, Mrs. Alexander (Helg…","""female""",22.0,1,1,"""3101298""",12.2875,null,"""S"""


In [30]:
# ============================================================
# RAW DATA STATISTICS
# ============================================================

print("TRAIN DESCRIBE")
display(train.describe())

print()

print("TEST DESCRIBE")
display(test.describe())

TRAIN DESCRIBE


statistic,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
str,f64,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",891.0,891.0,891.0,"""891""","""891""",714.0,891.0,891.0,"""891""",891.0,"""204""","""889"""
"""null_count""",0.0,0.0,0.0,"""0""","""0""",177.0,0.0,0.0,"""0""",0.0,"""687""","""2"""
"""mean""",446.0,0.383838,2.308642,null,null,29.699118,0.523008,0.381594,null,32.204208,null,null
"""std""",257.353842,0.486592,0.836071,null,null,14.526497,1.102743,0.806057,null,49.693429,null,null
"""min""",1.0,0.0,1.0,"""Abbing, Mr. Anthony""","""female""",0.42,0.0,0.0,"""110152""",0.0,"""A10""","""C"""
"""25%""",224.0,0.0,2.0,null,null,20.0,0.0,0.0,null,7.925,null,null
"""50%""",446.0,0.0,3.0,null,null,28.0,0.0,0.0,null,14.4542,null,null
"""75%""",669.0,1.0,3.0,null,null,38.0,1.0,0.0,null,31.0,null,null
"""max""",891.0,1.0,3.0,"""van Melkebeke, Mr. Philemon""","""male""",80.0,8.0,6.0,"""WE/P 5735""",512.3292,"""T""","""S"""



TEST DESCRIBE


statistic,passengerid,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
str,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",418.0,418.0,"""418""","""418""",332.0,418.0,418.0,"""418""",417.0,"""91""","""418"""
"""null_count""",0.0,0.0,"""0""","""0""",86.0,0.0,0.0,"""0""",1.0,"""327""","""0"""
"""mean""",1100.5,2.26555,null,null,30.27259,0.447368,0.392344,null,35.627188,null,null
"""std""",120.810458,0.841838,null,null,14.181209,0.89676,0.981429,null,55.907576,null,null
"""min""",892.0,1.0,"""Abbott, Master. Eugene Joseph""","""female""",0.17,0.0,0.0,"""110469""",0.0,"""A11""","""C"""
"""25%""",996.0,1.0,null,null,21.0,0.0,0.0,null,7.8958,null,null
"""50%""",1101.0,3.0,null,null,27.0,0.0,0.0,null,14.4542,null,null
"""75%""",1205.0,3.0,null,null,39.0,1.0,0.0,null,31.5,null,null
"""max""",1309.0,3.0,"""van Billiard, Master. Walter J…","""male""",76.0,8.0,9.0,"""W.E.P. 5734""",512.3292,"""G6""","""S"""


In [31]:
# ============================================================
# CARDINALITY PROFILING
# ============================================================

def cardinality_report(
    df: pl.DataFrame,
) -> pl.DataFrame:

    rows = []

    for column in df.columns:

        rows.append({
            "column": column,
            "unique_count": df[column].n_unique(),
            "null_count": df[column].null_count(),
        })

    return pl.DataFrame(rows)


train_cardinality = cardinality_report(train)
test_cardinality = cardinality_report(test)


print("TRAIN CARDINALITY")
display(train_cardinality)

print()

print("TEST CARDINALITY")
display(test_cardinality)

TRAIN CARDINALITY


column,unique_count,null_count
str,i64,i64
"""passengerid""",891,0
"""survived""",2,0
"""pclass""",3,0
"""name""",891,0
"""sex""",2,0
"""age""",89,177
"""sibsp""",7,0
"""parch""",7,0
"""ticket""",681,0



TEST CARDINALITY


column,unique_count,null_count
str,i64,i64
"""passengerid""",418,0
"""pclass""",3,0
"""name""",418,0
"""sex""",2,0
"""age""",80,86
"""sibsp""",7,0
"""parch""",8,0
"""ticket""",363,0
"""fare""",170,1


In [32]:
# ============================================================
# WRITE STAGED ARTIFACTS
# ============================================================

train.write_parquet(
    TRAIN_PARQUET
)

test.write_parquet(
    TEST_PARQUET
)


assert TRAIN_PARQUET.exists(), (
    "Train Parquet tidak berhasil dibuat"
)

assert TEST_PARQUET.exists(), (
    "Test Parquet tidak berhasil dibuat"
)


assert TRAIN_PARQUET.stat().st_size > 0
assert TEST_PARQUET.stat().st_size > 0


print("✓ Staged Parquet artifacts created")

✓ Staged Parquet artifacts created


In [33]:
# ============================================================
# STAGED ARTIFACT FINGERPRINT
# ============================================================

train_staged_sha256 = sha256_file(
    TRAIN_PARQUET
)

test_staged_sha256 = sha256_file(
    TEST_PARQUET
)


train_staged_size = (
    TRAIN_PARQUET.stat().st_size
)

test_staged_size = (
    TEST_PARQUET.stat().st_size
)


print("Train artifact SHA256:")
print(train_staged_sha256)

print()

print("Test artifact SHA256:")
print(test_staged_sha256)

Train artifact SHA256:
8057e012ff35427402e0e22e1227e691a2afa78e99595ed7c5c8b5bc9110c271

Test artifact SHA256:
aab5bc44778f5c903fa35786c16a4849533b4a5afff20e031575745a4eb3950c


In [34]:
# ============================================================
# STAGED ARTIFACT READ-BACK VALIDATION
# ============================================================

train_staged = pl.read_parquet(
    TRAIN_PARQUET
)

test_staged = pl.read_parquet(
    TEST_PARQUET
)


assert train_staged.columns == EXPECTED_TRAIN_COLUMNS

assert test_staged.columns == EXPECTED_TEST_COLUMNS


validate_dtypes(
    train_staged,
    "train_staged",
)

validate_dtypes(
    test_staged,
    "test_staged",
)


assert train_staged.height == train.height

assert test_staged.height == test.height


assert train_staged.width == train.width

assert test_staged.width == test.width


print("✓ Staged artifacts readable and structurally valid")

✓ Staged artifacts readable and structurally valid


In [35]:
# ============================================================
# STAGED SCHEMA FINGERPRINT
# ============================================================

train_staged_schema_sha256 = schema_fingerprint(
    train_staged
)

test_staged_schema_sha256 = schema_fingerprint(
    test_staged
)


assert train_staged_schema_sha256 == train_schema_sha256

assert test_staged_schema_sha256 == test_schema_sha256


print("✓ Staged schema fingerprints match source schema")

✓ Staged schema fingerprints match source schema


In [36]:
# ============================================================
# VALIDATION RESULT STRUCTURE
# ============================================================

validation_results = []


def add_validation_result(
    rule_id: str,
    check: str,
    dataset: str,
    status: str,
    severity: str,
    actual=None,
    expected=None,
    message: str = "",
):

    validation_results.append({
        "rule_id": rule_id,
        "check": check,
        "dataset": dataset,
        "status": status,
        "severity": severity,
        "actual": actual,
        "expected": expected,
        "message": message,
    })

In [37]:
# ============================================================
# VALIDATION SUMMARY
# ============================================================

# Pastikan validation_results kosong sebelum membangun summary.
validation_results = []


def add_validation_result(
    rule_id: str,
    check: str,
    dataset: str,
    status: str,
    severity: str,
    actual=None,
    expected=None,
    message: str = "",
):
    """
    Menambahkan hasil validation dengan struktur yang konsisten.

    actual dan expected selalu dikonversi menjadi string JSON
    agar Polars tidak mengalami schema inference conflict.
    """

    def serialize_value(value):
        if value is None:
            return None

        if isinstance(value, (dict, list, tuple, set)):
            if isinstance(value, set):
                value = sorted(value)

            elif isinstance(value, tuple):
                value = list(value)

            return json.dumps(
                value,
                ensure_ascii=False,
                sort_keys=True,
            )

        return str(value)

    validation_results.append({
        "rule_id": str(rule_id),
        "check": str(check),
        "dataset": str(dataset),
        "status": str(status),
        "severity": str(severity),
        "actual": serialize_value(actual),
        "expected": serialize_value(expected),
        "message": str(message),
    })


# ============================================================
# DQ-001 — SOURCE FILES
# ============================================================

add_validation_result(
    rule_id="DQ-001",
    check="source_files",
    dataset="all",
    status="PASS",
    severity="CRITICAL",
    actual="train.csv + test.csv",
    expected="existing non-empty CSV files",
    message="Source files exist and are non-empty.",
)


# ============================================================
# DQ-002 — SOURCE FINGERPRINT
# ============================================================

add_validation_result(
    rule_id="DQ-002",
    check="source_fingerprint",
    dataset="all",
    status="PASS",
    severity="CRITICAL",
    actual={
        "train_sha256": train_sha256,
        "test_sha256": test_sha256,
    },
    expected="valid SHA-256 fingerprint",
    message="Source SHA-256 fingerprints generated successfully.",
)


# ============================================================
# DQ-003 — TRAIN SCHEMA
# ============================================================

add_validation_result(
    rule_id="DQ-003",
    check="schema",
    dataset="train",
    status="PASS",
    severity="CRITICAL",
    actual=train.columns,
    expected=EXPECTED_TRAIN_COLUMNS,
    message="Train column schema matches the contract.",
)


# ============================================================
# DQ-004 — TEST SCHEMA
# ============================================================

add_validation_result(
    rule_id="DQ-004",
    check="schema",
    dataset="test",
    status="PASS",
    severity="CRITICAL",
    actual=test.columns,
    expected=EXPECTED_TEST_COLUMNS,
    message="Test column schema matches the contract.",
)


# ============================================================
# DQ-005 — DATA TYPES
# ============================================================

add_validation_result(
    rule_id="DQ-005",
    check="data_types",
    dataset="all",
    status="PASS",
    severity="CRITICAL",
    actual={
        "train": {
            column: str(dtype)
            for column, dtype in train.schema.items()
        },
        "test": {
            column: str(dtype)
            for column, dtype in test.schema.items()
        },
    },
    expected={
        column: str(dtype)
        for column, dtype in EXPECTED_DTYPES.items()
    },
    message="Data types match the schema contract.",
)


# ============================================================
# DQ-006 — ROW COUNT
# ============================================================

add_validation_result(
    rule_id="DQ-006",
    check="row_count",
    dataset="all",
    status="PASS",
    severity="CRITICAL",
    actual={
        "train": train.height,
        "test": test.height,
    },
    expected="train > 0 and test > 0",
    message="Both datasets contain at least one row.",
)


# ============================================================
# DQ-007 — PRIMARY KEY
# ============================================================

add_validation_result(
    rule_id="DQ-007",
    check="primary_key",
    dataset="all",
    status="PASS",
    severity="CRITICAL",
    actual={
        "train": "unique and non-null",
        "test": "unique and non-null",
    },
    expected="unique and non-null passengerid",
    message="Passenger identifiers are unique, non-null, and positive.",
)


# ============================================================
# DQ-008 — DUPLICATE ROWS
# ============================================================

add_validation_result(
    rule_id="DQ-008",
    check="duplicate_rows",
    dataset="all",
    status="PASS",
    severity="CRITICAL",
    actual={
        "train": train_duplicate_rows,
        "test": test_duplicate_rows,
    },
    expected=0,
    message="No full duplicate records detected.",
)


# ============================================================
# DQ-009 — MISSINGNESS
# ============================================================

add_validation_result(
    rule_id="DQ-009",
    check="missingness",
    dataset="all",
    status="PASS",
    severity="CRITICAL",
    actual="within configured thresholds",
    expected="within configured thresholds",
    message="Missingness is within the configured project thresholds.",
)


# ============================================================
# DQ-010 — STRING QUALITY
# ============================================================

string_warning_count = (
    train_string_quality.filter(
        pl.col("status") == "WARNING"
    ).height
    +
    test_string_quality.filter(
        pl.col("status") == "WARNING"
    ).height
)


string_fail_count = (
    train_string_quality.filter(
        pl.col("status") == "FAIL"
    ).height
    +
    test_string_quality.filter(
        pl.col("status") == "FAIL"
    ).height
)


if string_fail_count > 0:
    string_status = "FAIL"

elif string_warning_count > 0:
    string_status = "WARNING"

else:
    string_status = "PASS"


add_validation_result(
    rule_id="DQ-010",
    check="string_quality",
    dataset="all",
    status=string_status,
    severity="WARNING",
    actual={
        "warnings": string_warning_count,
        "failures": string_fail_count,
    },
    expected={
        "empty_strings": 0,
        "leading_trailing_whitespace": 0,
    },
    message=(
        "String quality checked without modifying raw values."
    ),
)


# ============================================================
# DQ-011 — NUMERIC DOMAIN
# ============================================================

add_validation_result(
    rule_id="DQ-011",
    check="numeric_domain",
    dataset="all",
    status="PASS",
    severity="CRITICAL",
    actual="validated",
    expected="valid configured numeric domains",
    message="Numeric domain validation passed.",
)


# ============================================================
# DQ-012 — CATEGORICAL DOMAIN
# ============================================================

add_validation_result(
    rule_id="DQ-012",
    check="categorical_domain",
    dataset="all",
    status="PASS",
    severity="CRITICAL",
    actual="validated",
    expected="configured allowed values",
    message="Categorical domain validation passed.",
)


# ============================================================
# DQ-013 — TRAIN / TEST RELATIONSHIP
# ============================================================

add_validation_result(
    rule_id="DQ-013",
    check="train_test_relationship",
    dataset="all",
    status="PASS",
    severity="CRITICAL",
    actual={
        "overlap_count": len(overlap_ids),
    },
    expected={
        "overlap_count": 0,
    },
    message="Train and test passenger IDs do not overlap.",
)


# ============================================================
# DQ-014 — STAGED ARTIFACTS
# ============================================================

add_validation_result(
    rule_id="DQ-014",
    check="staged_artifacts",
    dataset="all",
    status="PASS",
    severity="CRITICAL",
    actual={
        "train": {
            "readable": True,
            "rows": train_staged.height,
            "columns": train_staged.width,
            "schema_sha256": train_staged_schema_sha256,
        },
        "test": {
            "readable": True,
            "rows": test_staged.height,
            "columns": test_staged.width,
            "schema_sha256": test_staged_schema_sha256,
        },
    },
    expected="readable and schema-compatible staged artifacts",
    message="Staged Parquet artifacts passed read-back validation.",
)


# ============================================================
# CREATE POLARS VALIDATION DATAFRAME
# ============================================================

validation_df = pl.DataFrame(
    validation_results,
    schema={
        "rule_id": pl.String,
        "check": pl.String,
        "dataset": pl.String,
        "status": pl.String,
        "severity": pl.String,
        "actual": pl.String,
        "expected": pl.String,
        "message": pl.String,
    },
)


display(validation_df)

rule_id,check,dataset,status,severity,actual,expected,message
str,str,str,str,str,str,str,str
"""DQ-001""","""source_files""","""all""","""PASS""","""CRITICAL""","""train.csv + test.csv""","""existing non-empty CSV files""","""Source files exist and are non…"
"""DQ-002""","""source_fingerprint""","""all""","""PASS""","""CRITICAL""","""{""test_sha256"": ""56023b9948236…","""valid SHA-256 fingerprint""","""Source SHA-256 fingerprints ge…"
"""DQ-003""","""schema""","""train""","""PASS""","""CRITICAL""","""[""passengerid"", ""survived"", ""p…","""[""passengerid"", ""survived"", ""p…","""Train column schema matches th…"
"""DQ-004""","""schema""","""test""","""PASS""","""CRITICAL""","""[""passengerid"", ""pclass"", ""nam…","""[""passengerid"", ""pclass"", ""nam…","""Test column schema matches the…"
"""DQ-005""","""data_types""","""all""","""PASS""","""CRITICAL""","""{""test"": {""age"": ""Float64"", ""c…","""{""age"": ""Float64"", ""cabin"": ""S…","""Data types match the schema co…"
"""DQ-006""","""row_count""","""all""","""PASS""","""CRITICAL""","""{""test"": 418, ""train"": 891}""","""train > 0 and test > 0""","""Both datasets contain at least…"
"""DQ-007""","""primary_key""","""all""","""PASS""","""CRITICAL""","""{""test"": ""unique and non-null""…","""unique and non-null passengeri…","""Passenger identifiers are uniq…"
"""DQ-008""","""duplicate_rows""","""all""","""PASS""","""CRITICAL""","""{""test"": 0, ""train"": 0}""","""0""","""No full duplicate records dete…"
"""DQ-009""","""missingness""","""all""","""PASS""","""CRITICAL""","""within configured thresholds""","""within configured thresholds""","""Missingness is within the conf…"


In [38]:
# ============================================================
# OVERALL QUALITY GATE
# ============================================================

statuses = validation_df["status"].to_list()


if "FAIL" in statuses:

    overall_status = "FAIL"

elif "WARNING" in statuses:

    overall_status = "PASS_WITH_WARNINGS"

else:

    overall_status = "PASS"


print("=" * 72)
print("OVERALL INGESTION QUALITY GATE")
print("=" * 72)
print(overall_status)
print("=" * 72)

OVERALL INGESTION QUALITY GATE
PASS_WITH_WARNINGS


In [39]:
# ============================================================
# VALIDATION STATISTICS
# ============================================================

validation_counts = {
    "total_rules": validation_df.height,

    "passed": validation_df.filter(
        pl.col("status") == "PASS"
    ).height,

    "warnings": validation_df.filter(
        pl.col("status") == "WARNING"
    ).height,

    "failed": validation_df.filter(
        pl.col("status") == "FAIL"
    ).height,
}


print(
    json.dumps(
        validation_counts,
        indent=2,
    )
)

{
  "total_rules": 14,
  "passed": 13,
  "warnings": 1,
  "failed": 0
}


In [40]:
# ============================================================
# GIT PROVENANCE
# ============================================================

def get_git_info() -> dict:

    result = {
        "repository": None,
        "branch": None,
        "commit": None,
        "available": False,
    }

    try:

        repository = subprocess.run(
            [
                "git",
                "config",
                "--get",
                "remote.origin.url",
            ],
            capture_output=True,
            text=True,
            check=False,
        )

        branch = subprocess.run(
            [
                "git",
                "branch",
                "--show-current",
            ],
            capture_output=True,
            text=True,
            check=False,
        )

        commit = subprocess.run(
            [
                "git",
                "rev-parse",
                "HEAD",
            ],
            capture_output=True,
            text=True,
            check=False,
        )

        if commit.returncode == 0:

            result["available"] = True

            result["repository"] = (
                repository.stdout.strip()
                if repository.returncode == 0
                else None
            )

            result["branch"] = (
                branch.stdout.strip()
                if branch.returncode == 0
                else None
            )

            result["commit"] = (
                commit.stdout.strip()
            )

    except FileNotFoundError:

        pass

    return result


git_info = get_git_info()


print(
    json.dumps(
        git_info,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "repository": "https://github.com/prasetya-code/titanic_survival_analysis.git",
  "branch": "main",
  "commit": "9dfd9eac8684a6540ed23023441ca23eeca4779c",
  "available": true
}


In [41]:
# ============================================================
# ENVIRONMENT
# ============================================================

environment_info = {
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "polars": pl.__version__,
}


print(
    json.dumps(
        environment_info,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "python": "3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]",
  "python_executable": "d:\\GIT DATA\\titanic_survival_analysis\\win_env\\Scripts\\python.exe",
  "platform": "Windows-11-10.0.26200-SP0",
  "machine": "AMD64",
  "processor": "Intel64 Family 6 Model 126 Stepping 5, GenuineIntel",
  "polars": "1.44.2"
}


In [42]:
# ============================================================
# INGESTION DURATION
# ============================================================

INGESTION_END = time.perf_counter()

INGESTION_DURATION_SECONDS = round(
    INGESTION_END - INGESTION_START,
    4,
)


print(
    f"Ingestion duration: "
    f"{INGESTION_DURATION_SECONDS} seconds"
)

Ingestion duration: 2.0079 seconds


In [43]:
# ============================================================
# INGESTION METADATA
# ============================================================

ingestion_timestamp_utc = RUN_TIMESTAMP.isoformat()


metadata = {
    "run": {
        "run_id": RUN_ID,
        "started_at_utc": ingestion_timestamp_utc,
        "duration_seconds": INGESTION_DURATION_SECONDS,
    },

    "dataset": {
        "name": DATASET_NAME,
        "source_format": SOURCE_FORMAT,
        "artifact_format": ARTIFACT_FORMAT,
    },

    "pipeline": {
        "stage": PIPELINE_STAGE,
        "pipeline_version": PIPELINE_VERSION,
        "schema_version": SCHEMA_VERSION,
        "dq_rules_version": DQ_RULES_VERSION,
    },

    "source": {
        "train": {
            "path": str(TRAIN_RAW),
            "file_name": TRAIN_RAW.name,
            "size_bytes": train_file_info["size_bytes"],
            "sha256": train_sha256,
            "format": SOURCE_FORMAT,
            "encoding": train_csv_structure["encoding"],
            "delimiter": train_csv_structure["delimiter"],
        },

        "test": {
            "path": str(TEST_RAW),
            "file_name": TEST_RAW.name,
            "size_bytes": test_file_info["size_bytes"],
            "sha256": test_sha256,
            "format": SOURCE_FORMAT,
            "encoding": test_csv_structure["encoding"],
            "delimiter": test_csv_structure["delimiter"],
        },
    },

    "schema": {
        "train": {
            "columns": EXPECTED_TRAIN_COLUMNS,
            "fingerprint_sha256": train_schema_sha256,
        },

        "test": {
            "columns": EXPECTED_TEST_COLUMNS,
            "fingerprint_sha256": test_schema_sha256,
        },
    },

    "datasets": {
        "train": train_summary,
        "test": test_summary,
    },

    "artifacts": {
        "train": {
            "path": str(TRAIN_PARQUET),
            "format": ARTIFACT_FORMAT,
            "size_bytes": train_staged_size,
            "sha256": train_staged_sha256,
            "schema_sha256": train_staged_schema_sha256,
            "rows": train_staged.height,
            "columns": train_staged.width,
        },

        "test": {
            "path": str(TEST_PARQUET),
            "format": ARTIFACT_FORMAT,
            "size_bytes": test_staged_size,
            "sha256": test_staged_sha256,
            "schema_sha256": test_staged_schema_sha256,
            "rows": test_staged.height,
            "columns": test_staged.width,
        },
    },

    "validation": {
        "overall_status": overall_status,

        "statistics": validation_counts,

        "rules": validation_results,
    },

    "metrics": {
        "train_rows": train.height,
        "test_rows": test.height,

        "train_columns": train.width,
        "test_columns": test.width,

        "train_duplicate_rows": train_duplicate_rows,
        "test_duplicate_rows": test_duplicate_rows,

        "train_test_id_overlap": len(overlap_ids),

        "string_quality_warnings": string_warning_count,
        "string_quality_failures": string_fail_count,
    },

    "provenance": {
        "git": git_info,
    },

    "environment": environment_info,

    "status": overall_status,
}


INGESTION_METADATA.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print(
    "✓ Metadata written:",
    INGESTION_METADATA,
)

✓ Metadata written: ..\data\staged\metadata\ingestion_metadata.json


In [44]:
# ============================================================
# VALIDATION REPORT
# ============================================================

validation_report = {
    "run_id": RUN_ID,

    "dataset": DATASET_NAME,

    "pipeline_stage": PIPELINE_STAGE,

    "pipeline_version": PIPELINE_VERSION,

    "schema_version": SCHEMA_VERSION,

    "dq_rules_version": DQ_RULES_VERSION,

    "timestamp_utc": ingestion_timestamp_utc,

    "overall_status": overall_status,

    "statistics": validation_counts,

    "rules": validation_results,
}


VALIDATION_REPORT.write_text(
    json.dumps(
        validation_report,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print(
    "✓ Validation report written:",
    VALIDATION_REPORT,
)

✓ Validation report written: ..\data\staged\metadata\validation_report.json


In [45]:
# ============================================================
# FINAL INGESTION REPORT
# ============================================================

final_report = {
    "run_id": RUN_ID,

    "status": overall_status,

    "dataset": DATASET_NAME,

    "train": {
        "rows": train.height,
        "columns": train.width,
        "source_sha256": train_sha256,
        "artifact_sha256": train_staged_sha256,
    },

    "test": {
        "rows": test.height,
        "columns": test.width,
        "source_sha256": test_sha256,
        "artifact_sha256": test_staged_sha256,
    },

    "validation": validation_counts,

    "duration_seconds": INGESTION_DURATION_SECONDS,

    "metadata_path": str(
        INGESTION_METADATA
    ),

    "validation_report_path": str(
        VALIDATION_REPORT
    ),
}


print("=" * 72)
print("FINAL INGESTION REPORT")
print("=" * 72)

print(
    json.dumps(
        final_report,
        indent=2,
        ensure_ascii=False,
    )
)

print("=" * 72)
print(
    "INGESTION QUALITY GATE:",
    overall_status,
)
print("=" * 72)

FINAL INGESTION REPORT
{
  "run_id": "titanic-ingestion-20260917T034007Z",
  "status": "PASS_WITH_WARNINGS",
  "dataset": "titanic",
  "train": {
    "rows": 891,
    "columns": 12,
    "source_sha256": "7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f",
    "artifact_sha256": "8057e012ff35427402e0e22e1227e691a2afa78e99595ed7c5c8b5bc9110c271"
  },
  "test": {
    "rows": 418,
    "columns": 11,
    "source_sha256": "56023b9948236f3c7a1c9448fcf418b283e109ef177fa8c7e069158dd7dd52b2",
    "artifact_sha256": "aab5bc44778f5c903fa35786c16a4849533b4a5afff20e031575745a4eb3950c"
  },
  "validation": {
    "total_rules": 14,
    "passed": 13,
    "warnings": 1,
    "failed": 0
  },
  "duration_seconds": 2.0079,
  "metadata_path": "..\\data\\staged\\metadata\\ingestion_metadata.json",
  "validation_report_path": "..\\data\\staged\\metadata\\validation_report.json"
}
INGESTION QUALITY GATE: PASS_WITH_WARNINGS


In [46]:
# ============================================================
# FINAL QUALITY GATE
# ============================================================

if overall_status == "FAIL":

    raise RuntimeError(
        "INGESTION FAILED. "
        "Periksa validation_report.json untuk detail."
    )


print(
    "✓ INGESTION QUALITY GATE PASSED"
)

if overall_status == "PASS_WITH_WARNINGS":

    print(
        "⚠ Dataset dapat dilanjutkan ke tahap berikutnya "
        "dengan warning yang tercatat."
    )

else:

    print(
        "✓ Tidak ada warning pada ingestion."
    )

✓ INGESTION QUALITY GATE PASSED
⚠ Dataset dapat dilanjutkan ke tahap berikutnya dengan warning yang tercatat.


In [47]:
# ============================================================
# RESOURCE CLEANUP
# ============================================================

# Jangan melakukan cleanup sebelum:
# - validation selesai
# - artifact selesai dibuat
# - metadata selesai dibuat
# - final report selesai dibuat

objects_to_delete = [
    "train_staged",
    "test_staged",
    "train",
    "test",

    "train_missing",
    "test_missing",

    "train_missing_threshold",
    "test_missing_threshold",

    "train_string_quality",
    "test_string_quality",

    "train_cardinality",
    "test_cardinality",

    "validation_df",
]


deleted_objects = []

for object_name in objects_to_delete:

    if object_name in globals():

        del globals()[object_name]

        deleted_objects.append(object_name)


collected_objects = gc.collect()


print(
    "✓ Resource cleanup completed"
)

print(
    "Deleted objects:",
    deleted_objects,
)

print(
    "Garbage collected:",
    collected_objects,
)

✓ Resource cleanup completed
Deleted objects: ['train_staged', 'test_staged', 'train', 'test', 'train_missing', 'test_missing', 'train_missing_threshold', 'test_missing_threshold', 'train_string_quality', 'test_string_quality', 'train_cardinality', 'test_cardinality', 'validation_df']
Garbage collected: 109


In [48]:
# ============================================================
# PIPELINE COMPLETION
# ============================================================

print("=" * 72)
print("INGESTION PIPELINE COMPLETED")
print("=" * 72)

print("Run ID :", RUN_ID)

print()

print("Train artifact:")
print(TRAIN_PARQUET)

print()

print("Test artifact:")
print(TEST_PARQUET)

print()

print("Metadata:")
print(INGESTION_METADATA)

print()

print("Validation report:")
print(VALIDATION_REPORT)

print()

print("Final status:")
print(overall_status)

print("=" * 72)

INGESTION PIPELINE COMPLETED
Run ID : titanic-ingestion-20260917T034007Z

Train artifact:
..\data\staged\train.parquet

Test artifact:
..\data\staged\test.parquet

Metadata:
..\data\staged\metadata\ingestion_metadata.json

Validation report:
..\data\staged\metadata\validation_report.json

Final status:
PASS_WITH_WARNINGS
